# QEC Simulation with LocalASC

This walkthrough demonstrates a comlpete flow for QEC using the LocalASC scheduler for Syndrome extraction circuits: loading parity-check matrices, generating a valid CSS memory circuit, running a decoding experiment using `sinter`, and plotting the logical error rates.

## 1. Imports and Workspace Setup
First, we ensure the project root is in our path and import the necessary functions from the `circuit_builder`, `simulation`, and `visualization` modules.

In [ ]:
import sys
import numpy as np
from pathlib import Path

# Find the repository root
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "codes").is_dir() and (path / "src").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the LocalASC repository root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.LocalASC.scheduling.localASC_scheduler import generate_schedule
from src.LocalASC.circuit_builder.circuit_builder import build_CSS_memory_circuit
from src.LocalASC.simulation.sinter_experiment import run_sinter_experiment
from src.LocalASC.visualization.processing_and_plotting import plot_qec_results

## 2. Generate the Schedule and Circuit
We will use the `[[12, 4, 2]]` code as our example. We load the parity-check matrices, generate an optimal CNOT schedule, and build stim memory circuits for X and Z- basis.

In [ ]:
# load PCMs
code_dir = project_root / "codes" / "Table1_PCMs"
Hx = np.loadtxt(code_dir / "01_n12_k4_d2_Hx.txt", dtype=np.uint8)
Hz = np.loadtxt(code_dir / "01_n12_k4_d2_Hz.txt", dtype=np.uint8)

# generate schedule and create path for the schedule to be saved
schedule_path = project_root / "results" / "schedules" / "tutorial_12_4.csv"
schedule_path.parent.mkdir(parents=True, exist_ok=True)

success, status = generate_schedule(Hx, Hz, D=6, output_csv=str(schedule_path),solver_tlim=60)

print(f"Schedule generation: {status}")

# build and save the circuits
rounds = 2
circuit_z = build_CSS_memory_circuit(Hx, Hz, schedule_path, basis="Z", rounds=rounds)
circuit_x = build_CSS_memory_circuit(Hx, Hz, schedule_path, basis="X", rounds=rounds)

## 3. Run the Sinter Decoding Experiment
With our circuit saved, we use the `sinter_experiment` module to sample the circuit across a range of physical error rates ($p$). The experiment output is saved as a JSON file.

In [ ]:
# The results are saved to the selected file
experiment_output = project_root / "results" / "sinter_data"/ "Code_n12_k4" / "tutorial_12_4_results.json"
experiment_output.parent.mkdir(parents=True, exist_ok=True)

experiment_output_dir = project_root / "results" / "sinter_data" / "Code_n12_k4"


# define the physical error rates to go over
physical_error_rates = np.logspace(-4, -2, 10).tolist()

# Run the Sinter sampling experiment
run_sinter_experiment(
    base_circ_x=circuit_x,
    base_circ_z=circuit_z,
    physical_error_probs= physical_error_rates,
    filename="tutorial_12_4_results.json",
    name="12_4_2", #Only required for json metadata
    n=12,
    k=4,
    d=2,
    nrounds=2,
    max_errors=100,
    max_bp_iter=1000,
    num_shots=10_000,
    output_dir= experiment_output_dir,
    verbose = False
    )

print(f"Experiment finished. Data saved to {experiment_output.relative_to(project_root)}")

## 4. Plot the Results
Finally, we use the plotting tools from `src.LocalASC.visualization.plotting` to read the generated JSON data, compute the pseudo-threshold, and visualize the logical error rate curve.

In [ ]:
# Configure the plot: (filepath, color, marker, label, distance)
code_distance = 2 
file_configs = [
    (experiment_output, "#C44E52", "o", "12_4 Code", code_distance)
]

plot_save_path = project_root / "results" / "plots" / "tutorial_12_4_ler_curve.jpg"

# Generate the plot
plot_qec_results(
    file_configs=file_configs,
    save_path=plot_save_path
)